# Global Life Expectancy Paradox — Data Cleaning Process

## Purpose

This notebook documents the **data-cleaning process used for the Global Life Expectancy Paradox project**.

The cleaning process documented in the project report consists of:

1. Loading and inspecting the dataset
2. Checking missing values
3. Filling missing categorical values using the **mode**
4. Filling missing numerical values using the **median**
5. Detecting exact duplicate rows
6. Removing duplicate rows
7. Validating the final dataset
8. Saving the final cleaned dataset

### Final dataset expected after cleaning

- **Rows:** 192
- **Columns:** 20
- **Missing cells:** 0
- **Duplicate rows:** 0

> **Important:** The report describes an original dataset containing 154 missing cells. The currently supplied pre-final file available with this project already has those missing values filled but still contains the 12 duplicate rows. Therefore, this notebook contains both:
> - a **full cleaning template** that can be run on the original raw dataset, and
> - a **direct final-cleaning section** that reproduces the current 192-row final dataset by removing the 12 remaining duplicates.


## 1. Import the required libraries

We use:

- `pandas` for reading, cleaning, inspecting, and saving tabular data.
- `numpy` for numerical operations where required.
- `Path` for convenient file-path handling.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Load the dataset

The file supplied with the project is named with an `.xls` extension, but its contents are actually CSV text. Therefore, it should be loaded with `pd.read_csv()` rather than `pd.read_excel()`.

For a true raw dataset containing the original missing values, change `INPUT_FILE` to that raw CSV file.


In [ ]:
# Current project file: 204 rows, 20 columns, 12 duplicate rows
INPUT_FILE = "CLEANED_DATASET (1).csv(3).xls"

df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Initial data inspection

Before changing anything, inspect:

- number of rows and columns
- column names
- data types
- missing values
- duplicate rows

This is important because we should understand the condition of the data before cleaning it.


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nTotal missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))


## 4. Missing-value cleaning

According to the project report, the original dataset contained **154 missing cells**.

The report specifies:

### Categorical variable
`iso2` → fill the missing value using the **mode**.

The mode is the most frequently occurring category.

### Numerical variables
The missing values in numerical columns were filled using the **median**.

The median is the middle value after sorting the non-missing observations. It is useful for this dataset because several variables can contain skewed values and large outliers.

### Why not simply delete the rows?

Deleting rows with missing values would reduce the number of country records. Median/mode imputation keeps the observations available for analysis.


In [ ]:
# ---- FULL MISSING-VALUE CLEANING TEMPLATE ----
# Run this section on the ORIGINAL raw dataset containing the 154 missing cells.

cleaned = df.copy()

# 1. Fill categorical missing values with the mode
categorical_columns = ["iso2"]

for col in categorical_columns:
    if cleaned[col].isna().any():
        mode_value = cleaned[col].mode(dropna=True)[0]
        cleaned[col] = cleaned[col].fillna(mode_value)
        print(f"{col}: missing values filled with mode = {mode_value}")

# 2. Fill numerical missing values with the median
numeric_columns = cleaned.select_dtypes(include=np.number).columns

for col in numeric_columns:
    missing_count = cleaned[col].isna().sum()
    if missing_count > 0:
        median_value = cleaned[col].median()
        cleaned[col] = cleaned[col].fillna(median_value)
        print(f"{col}: {missing_count} missing value(s) filled with median = {median_value}")

print("\nMissing cells after imputation:", int(cleaned.isna().sum().sum()))


## 5. Verify the imputation

After imputation, the number of missing cells should be **0**.

The project report records these original missing-value counts:

| Variable | Missing values |
|---|---:|
| `co2_emissions` | 59 |
| `homicide_rate` | 23 |
| `secondary_school_enrollment_male` | 14 |
| `secondary_school_enrollment_female` | 14 |
| `unemployment` | 10 |
| `infant_mortality` | 8 |
| `life_expectancy_male` | 6 |
| `life_expectancy_female` | 6 |
| `fertility` | 5 |
| `forested_area` | 4 |
| `internet_users` | 2 |
| `gdp_growth` | 1 |
| `iso2` | 1 |
| `gdp_per_capita` | 1 |

Total = **154 missing cells**.


In [ ]:
missing_summary = cleaned.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Total missing cells:", int(cleaned.isna().sum().sum()))

if missing_summary.empty:
    print("✓ No missing values remain.")
else:
    display(missing_summary.to_frame("remaining_missing_values"))


## 6. Detect duplicate rows

A duplicate row is a row whose complete set of values is identical to another row.

`df.duplicated()` returns `True` for duplicate rows (by default, keeping the first occurrence).

The supplied pre-final dataset contains **12 duplicate rows**, so these records need to be removed.


In [ ]:
# Show how many duplicate rows exist
duplicate_count = cleaned.duplicated().sum()
print("Duplicate rows:", int(duplicate_count))

# Display all copies of duplicated records
duplicates = cleaned[cleaned.duplicated(keep=False)].sort_values(by="name")
display(duplicates)


## 7. Remove duplicate rows

`drop_duplicates()` keeps the first occurrence of each exact duplicate and removes the repeated copies.

Because the project contains country-level records, retaining one copy of an identical country record prevents repeated observations from artificially influencing statistics and models.


In [ ]:
before_rows = len(cleaned)

cleaned = cleaned.drop_duplicates().reset_index(drop=True)

after_rows = len(cleaned)

print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)
print("Rows removed:", before_rows - after_rows)


## 8. Final validation

The final dataset should satisfy all four conditions:

- 192 rows
- 20 columns
- 0 missing cells
- 0 duplicate rows

This validation step is important because cleaning is not complete until the result is checked.


In [ ]:
final_missing = int(cleaned.isna().sum().sum())
final_duplicates = int(cleaned.duplicated().sum())

print("FINAL DATASET CHECK")
print("-" * 30)
print("Rows:", cleaned.shape[0])
print("Columns:", cleaned.shape[1])
print("Missing cells:", final_missing)
print("Duplicate rows:", final_duplicates)

assert cleaned.shape == (192, 20), "Check the input dataset or cleaning steps."
assert final_missing == 0, "Missing values still remain."
assert final_duplicates == 0, "Duplicate rows still remain."

print("\n✓ All final validation checks passed.")


## 9. Save the final cleaned dataset

The cleaned data can be saved as either CSV or Excel.

CSV is convenient for programming and data-analysis workflows, while Excel is convenient for manual inspection and presentation.


In [ ]:
OUTPUT_CSV = "CLEANED_DATASET_FINAL.csv"
OUTPUT_XLSX = "CLEANED_DATASET_FINAL.xlsx"

cleaned.to_csv(OUTPUT_CSV, index=False)
cleaned.to_excel(OUTPUT_XLSX, index=False)

print("Saved:")
print(OUTPUT_CSV)
print(OUTPUT_XLSX)


# 10. Cleaning summary

### Before cleaning

The project report describes:

- 204 records
- 20 variables
- 154 missing cells
- 12 duplicate rows

### Cleaning operations

**Missing values**
- `iso2` → mode
- numerical variables → median

**Duplicates**
- 12 exact duplicate rows → removed

### Final dataset

- **192 records**
- **20 variables**
- **0 missing cells**
- **0 duplicate rows**

The cleaned dataset is then ready for the next stages of the project: exploratory data analysis, correlation analysis, visualization, regression, and investigation of the global life expectancy paradox.


# Code explained in simple terms

## `df.isna().sum()`

Counts missing values in each column.

```python
df.isna().sum()
```

## `df.isna().sum().sum()`

Counts all missing cells in the entire dataset.

```python
df.isna().sum().sum()
```

The first `sum()` adds values within each column; the second adds the column totals.

---

## `df.duplicated()`

Checks whether each row is an exact duplicate of an earlier row.

```python
df.duplicated()
```

---

## `df.drop_duplicates()`

Removes duplicate rows.

```python
df = df.drop_duplicates()
```

---

## `df.mode()`

Finds the most frequently occurring value.

```python
df["iso2"].mode()
```

This is used for the missing categorical value.

---

## `df.median()`

Calculates the median of a numerical column.

```python
df["gdp_per_capita"].median()
```

---

## `fillna()`

Replaces missing values.

```python
df["gdp_per_capita"] = df["gdp_per_capita"].fillna(
    df["gdp_per_capita"].median()
)
```

---

## `select_dtypes(include=np.number)`

Selects only numerical columns.

```python
numeric_columns = df.select_dtypes(include=np.number).columns
```

This lets us apply median imputation only to numerical variables.

---

## `reset_index(drop=True)`

After deleting duplicate rows, the original row numbers may have gaps. This recreates a clean sequential index.

```python
df = df.drop_duplicates().reset_index(drop=True)
```

---

## `to_csv()`

Saves the DataFrame as a CSV file.

```python
df.to_csv("cleaned.csv", index=False)
```

`index=False` prevents pandas from adding the DataFrame index as an extra column.

---

## `to_excel()`

Saves the DataFrame as an Excel workbook.

```python
df.to_excel("cleaned.xlsx", index=False)
```

---

## `assert`

Checks that a condition is true.

For example:

```python
assert final_missing == 0
```

If missing values still exist, Python stops and reports an error. If the condition is true, execution continues.


# Important note about this notebook

The **154 missing values are documented in the cleaning report**, but the currently supplied 204-row file available for this notebook already has those missing values filled. It contains 204 rows and 12 duplicates.

Therefore:

- If you run the notebook using the supplied 204-row file, the missing-value section will report **0 missing cells**, and the duplicate-removal section will reduce the data from **204 to 192 rows**.
- If you have the original raw dataset with the 154 missing cells, use that file as `INPUT_FILE`; the imputation code will perform the complete missing-value cleaning described in the report, followed by duplicate removal.

This distinction is intentional so the notebook does not invent or reconstruct raw values that are not present in the currently supplied file.
